1. Imports and basic setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import glob
import numpy as np
import pandas as pd

import tifffile as tiff
from skimage.measure import regionprops_table

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import HuberRegressor
from scipy.stats import zscore

sns.set(style="whitegrid", context="talk")

2. Helper: load multichannel 3D TIFF

Assume TIFF shape is either:
(Z, C, Y, X) or
(C, Z, Y, X)
We standardize to (C, Z, Y, X).

In [ ]:
def load_tiff(path):
    img = tiff.imread(path)

    if img.ndim != 4:
        raise ValueError(f"Expected 4D TIFF, got shape {img.shape}")

    # Heuristic: channel dimension is the smallest
    channel_axis = np.argmin(img.shape)

    if channel_axis != 0:
        img = np.moveaxis(img, channel_axis, 0)

    print(f'image shape : {img.shape}')
    return img  # shape: (C, Z, Y, X)


3. Feature extraction per nucleus (core step)

We compute:

Nuclear volume
Integrated DAPI intensity
Region label
Tissue annotation (Tumor/Adjacent)

In [ ]:

def extract_nuclei_features(
    label_img: np.ndarray,
    dapi_img: np.ndarray,
    region_mask: np.ndarray,
    dataset_type: str = "control",
    sample_name: str | None = None,
    *,
    keep_mask_value: bool = False,
) -> pd.DataFrame | tuple[pd.DataFrame, pd.DataFrame]:
    """
    Parameters
    ----------
    label_img : 3D ndarray
        Labeled nuclei image (one integer per nucleus).
    dapi_img : 3D ndarray
        DAPI intensity image.
    region_mask : 3D ndarray
        Mask image:
          - control: 100 = CV, 200 = PV, 0 = parenchyma
          - HCC:     >0 = tumor, 0 = adjacent
    dataset_type : str
        One of ["control", "hcc"].

    Returns
    -------
    For dataset_type == "control":
        df_control : pandas.DataFrame
    For dataset_type == "hcc":
        (df_adjacent, df_tumor) : tuple[pandas.DataFrame, pandas.DataFrame]
    """

    if label_img.shape != dapi_img.shape or label_img.shape != region_mask.shape:
        raise ValueError(
            f"Shape mismatch: label_img{label_img.shape}, dapi_img{dapi_img.shape}, region_mask{region_mask.shape}"
        )

    dataset_type = str(dataset_type).lower().strip()
    if dataset_type not in {"control", "hcc"}:
        raise ValueError("dataset_type must be one of ['control', 'hcc']")

    # --- extract regionprops ---
    props = regionprops_table(
        label_img,
        intensity_image=dapi_img,
        properties=["label", "area", "intensity_mean", "centroid"],
    )
    df = pd.DataFrame(props).rename(
        columns={
            "label": "nucleus_label",
            "area": "volume",          # voxel count
            "intensity_mean": "dapi_mean",
        }
    )

    # Integrated intensity (sum approx = mean * voxel_count)
    df["dapi_sum"] = df["dapi_mean"] * df["volume"]

    # --- centroid coordinates -> mask lookup ---
    centroid_cols = ("centroid-0", "centroid-1", "centroid-2")
    missing = [c for c in centroid_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing centroid columns from regionprops_table: {missing}")

    zc = np.clip(np.rint(df["centroid-0"].to_numpy()).astype(int), 0, region_mask.shape[0] - 1)
    yc = np.clip(np.rint(df["centroid-1"].to_numpy()).astype(int), 0, region_mask.shape[1] - 1)
    xc = np.clip(np.rint(df["centroid-2"].to_numpy()).astype(int), 0, region_mask.shape[2] - 1)

    mask_val = region_mask[zc, yc, xc]
    if keep_mask_value:
        df["mask_value"] = mask_val

    # metadata
    df["sample"] = sample_name

    # --- dataset-specific filtering / outputs ---
    if dataset_type == "control":
        keep = mask_val == 0  # parenchyma only
        out = df.loc[keep].copy()
        out["condition"] = "Control"
        out["tissue_class"] = "Parenchyma"
        return out

    # dataset_type == "hcc"
    keep_adjacent = mask_val == 0
    keep_tumor = mask_val > 0

    df_adjacent = df.loc[keep_adjacent].copy()
    df_adjacent["condition"] = "Adjacent"
    df_adjacent["tissue_class"] = "Adjacent"

    df_tumor = df.loc[keep_tumor].copy()
    df_tumor["condition"] = "Tumor"
    df_tumor["tissue_class"] = "Tumor"

    return df_adjacent, df_tumor



4. Loop over folders and build master dataframe

In [ ]:
def process_folder(folder, region_type):
    all_dfs = []

    for path in glob.glob(os.path.join(folder, "*.tif")):
        print(f"Processing: {path}")
        img = load_tiff(path)

        labels = img[0]
        dapi = img[1]
        region_mask = img[2]

        df = extract_nuclei_features(
            labels,
            dapi,
            region_mask,
            dataset_type=region_type,
            sample_name = os.path.basename(path)
        )

        all_dfs.append(df)

    return pd.concat(all_dfs, ignore_index=True)


In [ ]:
adj_df, tumor_df = process_folder("/content/drive/MyDrive/Projects/NucVerse/HCC/", "hcc")
controls_df = process_folder("/content/drive/MyDrive/Projects/NucVerse/Control/", "control")

df = pd.concat([controls_df, adj_df, tumor_df], ignore_index=True)


Processing: /content/drive/MyDrive/Projects/NucVerse/HCC/Composite_stam_13_16w.tif
image shape : (3, 353, 1146, 2467)
Processing: /content/drive/MyDrive/Projects/NucVerse/HCC/Composite_stam_8_16w.tif


5. Log₂ transform features

In [ ]:
df = df[(df["volume"] > 0) & (df["dapi"] > 0)]

df["log2_volume"] = np.log2(df["volume"])
df["log2_dapi"] = np.log2(df["dapi"])

6. Fit Huber regression on controls only

In [ ]:
ctrl = df[df["condition"] == "control"]

X = ctrl[["log2_volume"]].values
y = ctrl["log2_dapi"].values

huber = HuberRegressor(epsilon=1.35)
huber.fit(X, y)

alpha = huber.intercept_
beta = huber.coef_[0]

print(f"Reference model: log2(DAPI) = {alpha:.3f} + {beta:.3f} * log2(volume)")


7. Compute residuals and NDS

In [ ]:
df["expected_log2_dapi"] = alpha + beta * df["log2_volume"]
df["residual"] = df["log2_dapi"] - df["expected_log2_dapi"]

sigma_control = ctrl["residual"].std()

df["NDS"] = np.abs(df["residual"]) / sigma_control
df["sNDS"] = df["residual"] / sigma_control


In [ ]:
#Residual distributions (most important diagnostic)
#Why
#Control residuals should be centered at 0
#Adjacent slightly broader
#Tumor skewed / heavy-tailed

plt.figure(figsize=(6, 4))

sns.kdeplot(
    data=df,
    x="residual",
    hue="condition",
    common_norm=False,
    linewidth=2
)

plt.axvline(0, linestyle="--", color="black", linewidth=1)
plt.xlabel("Residual (log2 DAPI − expected)")
plt.ylabel("Density")
plt.title("Residual distribution by tissue type")
plt.legend(title="Condition")

plt.tight_layout()
plt.show()


In [ ]:
#Standardized residuals (sNDS)
#This should be unit variance in controls
#Makes conditions directly comparable

plt.figure(figsize=(6, 4))

sns.kdeplot(
    data=df,
    x="sNDS",
    hue="condition",
    common_norm=False,
    linewidth=2
)

plt.axvline(0, linestyle="--", color="black", linewidth=1)
plt.xlabel("Signed NDS (sNDS)")
plt.ylabel("Density")
plt.title("Signed nuclear decoupling score")
plt.legend(title="Condition")

plt.tight_layout()
plt.show()


In [ ]:
#Absolute NDS distributions (abnormality score)
#This is your actual marker
#Should show clear separation

plt.figure(figsize=(6, 4))

sns.kdeplot(
    data=df,
    x="NDS",
    hue="condition",
    common_norm=False,
    linewidth=2,
    clip=(0, None)
)

plt.xlabel("NDS (|residual| / σ_control)")
plt.ylabel("Density")
plt.title("Nuclear abnormality score (NDS)")
plt.legend(title="Condition")

plt.tight_layout()
plt.show()


In [ ]:
#Boxplot summary
plt.figure(figsize=(4, 4))

sns.boxplot(
    data=df,
    x="condition",
    y="NDS",
    showfliers=False
)

plt.axhline(2, linestyle="--", color="red", linewidth=1)
plt.ylabel("NDS")
plt.xlabel("")
plt.title("Nuclear decoupling score by tissue")
plt.legend(title="Condition")

plt.tight_layout()
plt.show()


In [ ]:
#Residual vs volume (checks model validity)
#Ensures no volume-dependent bias
#Especially important for tumors

plt.figure(figsize=(6, 4))

sns.scatterplot(
    data=df.sample(5000) if len(df) > 5000 else df,
    x="log2_volume",
    y="residual",
    hue="condition",
    alpha=0.3,
    linewidth=0
)

plt.axhline(0, linestyle="--", color="black")
plt.xlabel("log2 nuclear volume")
plt.ylabel("Residual")
plt.title("Residuals vs nuclear volume")
plt.legend(title="Condition")

plt.tight_layout()
plt.show()


In [ ]:
#ECDF (very robust)
#Great for comparing tails.

plt.figure(figsize=(6, 4))

sns.ecdfplot(
    data=df,
    x="NDS",
    hue="condition"
)

plt.xlabel("NDS")
plt.ylabel("Cumulative fraction")
plt.title("Cumulative distribution of NDS")
plt.legend(title="Condition")

plt.tight_layout()
plt.show()

In [ ]:
# Minimal QC table

df.groupby("condition")[["residual", "sNDS", "NDS"]].agg(
    mean_residual=("residual", "mean"),
    std_residual=("residual", "std"),
    mean_sNDS=("sNDS", "mean"),
    frac_NDS_gt2=("NDS", lambda x: (x > 2).mean()),
    frac_NDS_gt3=("NDS", lambda x: (x > 3).mean())
)


8. Classify nuclei

In [ ]:
def classify_nds(nds):
    if nds < 1:
        return "Normal"
    elif nds < 2:
        return "Mild"
    elif nds < 3:
        return "Abnormal"
    else:
        return "Strongly abnormal"

df["nds_class"] = df["NDS"].apply(classify_nds)
